In [10]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
def load_data(document):
    file_paths=Path(document)
    files=list(file_paths.glob("**/*pdf"))
    all_files=[]
    for i in files:
        print(f"processing {i}")
        try:
            doc=PyPDFLoader(str(i))
            data=doc.load()
            all_files.extend(data)
        except Exception as e:
            print(f"cannot open{i} due to {e}")
    return all_files

In [3]:
documents=load_data(r"E:\Datasets\land_slide_pdf")

processing E:\Datasets\land_slide_pdf\LandslideAtlas_new_2023.pdf
processing E:\Datasets\land_slide_pdf\Landslides_in_India_Issues_and_Perspective.pdf
processing E:\Datasets\land_slide_pdf\Landslide_Preparedness_Guide_.pdf


In [4]:
def text_split(document,chunk_size=1000,chunk_overlap=200):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n","\n","."],
        length_function=len
    )
    doc=splitter.split_documents(document)
    return doc

In [5]:
after_split=text_split(documents)

In [6]:
def create_db(doc):
    dir="chroma_db"
    embedding=HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_space=Chroma.from_documents(
        documents=doc,
        embedding=embedding,
        persist_directory=dir
    )
    return vector_space

In [7]:
vector_space=create_db(after_split)

C:\Users\user\AppData\Local\Temp\ipykernel_9628\1264843497.py:3: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceBgeEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7619.82it/s]


In [12]:
ret=vector_space.as_retriever()
ret.invoke("land slide")

[Document(id='e649a63c-2d91-4657-b5fe-01629e86c297', metadata={'author': 'jain_nirmala', 'moddate': '2024-08-05T16:07:48+05:30', 'creationdate': '2024-08-05T16:07:48+05:30', 'page': 86, 'total_pages': 93, 'source': 'E:\\Datasets\\land_slide_pdf\\LandslideAtlas_new_2023.pdf', 'page_label': '87', 'producer': 'Microsoft® PowerPoint® LTSC', 'creator': 'Microsoft® PowerPoint® LTSC', 'title': 'Indian Landslides Atlas'}, page_content='object-oriented methods. Geomorphology, Volume 116, 2010, 24-36.\nhttps://doi.org/10.1016/j.geomorph.2009.10.004.\nTapas Ranjan Martha, Norman Kerle, Cees J. van Westen, Victor Jetten, K. Vinod Kumar. Object-oriented\nanalysis of multi-temporal panchromatic images for creation of historical landslide inventories,\nISPRS Journal of Photogrammetry and Remote Sensing, Volume 67, 2012, 105-119,\nhttps://doi.org/10.1016/j.isprsjprs.2011.11.004.\nTurner, A. Keith, and Schuster, L. Robert. Landslides—Investigation and mitigation. National Research\nCouncil, National Ac

In [15]:
def rag_pipeline(vector_space,question):
    prompt=PromptTemplate(
        input_variables=["context","query"],
        template="this is the context:{context} answer from thi context only,if the anwer is not present then return not available in document,the question is {query},Answer:"
    )
    model=init_chat_model(model="groq:openai/gpt-oss-120b")
    retriver=vector_space.as_retriever()
    document=retriver.invoke(question)
    all_document="\n\n".join([d.page_content for d in document]) if document else "Not present in document"
    chain=prompt | model | StrOutputParser()
    return chain.invoke({"context":all_document,"query":question})


In [17]:
print(rag_pipeline(vector_space,"why dsa?"))

Not available in document.
